#  Análise de Pipeline de Vendas B2B - OptiSuite

---

##  Contexto do Negócio

**Empresa:** OptiSuite (fictícia - dataset público)  
**Segmento:** Software B2B (SaaS) - Plataforma de gestão empresarial  
**Período Analisado:** Outubro/2016 - Dezembro/2017 (15 meses)

---

##  Conjunto de Dados

 **sales_pipeline**  8.800  Oportunidades de venda (deals) 

 **products**  7  Catálogo de produtos (3 linhas: GTX, MG, GTK) 

 **accounts**  85  Clientes corporativos B2B 

 **sales_teams**  35  Vendedores e gerentes (3 regiões) 

---

##  Objetivos do Projeto

1. **Fase 1 - Python/Pandas:** Limpeza, exploração e feature engineering
2. **Fase 2 - SQL/PostgreSQL:** Modelagem dimensional (star schema) e análises
3. **Fase 3 - Power BI:** Dashboards interativos e visualizações



##  Resultados Principais

- **Taxa de Conversão:** 48.16% 
- **Receita Total:** $10.01M
- **Ciclo Médio de Venda:** 52 dias
- **Produto Campeão:** GTX Pro ($3.5M - 35% da receita)


##  Sumário

1. Importação de Bibliotecas
2. Carregamento dos Dados
3. Exploração Inicial (EDA)
4. Limpeza e Tratamento
5. Feature Engineering
6. Análises Descritivas
7. Preparação para SQL
8. Conclusões

---


##  IMPORTAÇÃO DE BIBLIOTECAS

Bibliotecas utilizadas para manipulação, análise e visualização de dados.

In [23]:
# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Datas
from datetime import datetime

# Configurações
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)



## CARREGAMENTO DOS DADOS

Carregando os 4 arquivos CSV do dataset original.

In [24]:
# Carregar datasets
sales_pipeline = pd.read_csv('sales_pipeline.csv')
products = pd.read_csv('products.csv')
sales_teams = pd.read_csv('sales_teams.csv')
accounts = pd.read_csv('accounts.csv')

# Visualizar dimensões
print(f"Sales Pipeline: {sales_pipeline.shape[0]:,} linhas × {sales_pipeline.shape[1]} colunas")
print(f"Products:       {products.shape[0]:,} linhas × {products.shape[1]} colunas")
print(f"Sales Teams:    {sales_teams.shape[0]:,} linhas × {sales_teams.shape[1]} colunas")
print(f"Accounts:       {accounts.shape[0]:,} linhas × {accounts.shape[1]} colunas")

Sales Pipeline: 8,800 linhas × 8 colunas
Products:       7 linhas × 3 colunas
Sales Teams:    35 linhas × 3 colunas
Accounts:       85 linhas × 7 colunas



## EXPLORAÇÃO INICIAL (EDA)

### Visão Geral das Tabelas

In [25]:
# Informações gerais de cada tabela
print("SALES PIPELINE")
print(sales_pipeline.info())
print("Primeiras 5 linhas:")
display(sales_pipeline.head())

print("\n")
print("PRODUCTS")
print(products.info())
display(products.head())

print("\n")
print("SALES TEAMS")
print(sales_teams.info())
display(sales_teams.head())

print("\n")
print("ACCOUNTS")
print(accounts.info())
display(accounts.head())

SALES PIPELINE
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8800 entries, 0 to 8799
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   opportunity_id  8800 non-null   object 
 1   sales_agent     8800 non-null   object 
 2   product         8800 non-null   object 
 3   account         8800 non-null   object 
 4   deal_stage      8800 non-null   object 
 5   engage_date     8300 non-null   object 
 6   close_date      6711 non-null   object 
 7   close_value     6711 non-null   float64
dtypes: float64(1), object(7)
memory usage: 550.1+ KB
None
Primeiras 5 linhas:


,opportunity_id,sales_agent,product,account,deal_stage,engage_date,close_date,close_value
0,1C1I7A6R,Moses Frase,GTX Plus Basic,Cancity,Won,2016-10-20,2017-03-01,1054.0
1,Z063OYW0,Darcel Schlecht,GTXPro,Isdom,Won,2016-10-25,2017-03-11,4514.0
2,EC4QE1BX,Darcel Schlecht,MG Special,Cancity,Won,2016-10-25,2017-03-07,50.0
3,MV1LWRNH,Moses Frase,GTX Basic,Codehow,Won,2016-10-25,2017-03-09,588.0
4,PE84CX4O,Zane Levy,GTX Basic,Hatfan,Won,2016-10-25,2017-03-02,517.0




PRODUCTS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   product      7 non-null      object
 1   series       7 non-null      object
 2   sales_price  7 non-null      int64 
dtypes: int64(1), object(2)
memory usage: 300.0+ bytes
None


,product,series,sales_price
0,GTX Basic,GTX,550
1,GTX Pro,GTX,4821
2,MG Special,MG,55
3,MG Advanced,MG,3393
4,GTX Plus Pro,GTX,5482




SALES TEAMS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   sales_agent      35 non-null     object
 1   manager          35 non-null     object
 2   regional_office  35 non-null     object
dtypes: object(3)
memory usage: 972.0+ bytes
None


,sales_agent,manager,regional_office
0,Anna Snelling,Dustin Brinkmann,Central
1,Cecily Lampkin,Dustin Brinkmann,Central
2,Versie Hillebrand,Dustin Brinkmann,Central
3,Lajuana Vencill,Dustin Brinkmann,Central
4,Moses Frase,Dustin Brinkmann,Central




ACCOUNTS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85 entries, 0 to 84
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   account           85 non-null     object 
 1   sector            85 non-null     object 
 2   year_established  85 non-null     int64  
 3   revenue           85 non-null     float64
 4   employees         85 non-null     int64  
 5   office_location   85 non-null     object 
 6   subsidiary_of     15 non-null     object 
dtypes: float64(1), int64(2), object(4)
memory usage: 4.8+ KB
None


,account,sector,year_established,revenue,employees,office_location,subsidiary_of
0,Acme Corporation,technolgy,1996,1100.04,2822,United States,NaN
1,Betasoloin,medical,1999,251.41,495,United States,NaN
2,Betatech,medical,1986,647.18,1185,Kenya,NaN
3,Bioholding,medical,2012,587.34,1356,Philipines,NaN
4,Bioplex,medical,1991,326.82,1016,United States,NaN


### Análise de Valores Ausentes

In [26]:
# Verificar valores nulos

tabelas = {
    'Sales Pipeline': sales_pipeline,
    'Products': products,
    'Sales Teams': sales_teams,
    'Accounts': accounts
}

for nome, df in tabelas.items():
    print(f"\n{nome}:")
    nulos = df.isnull().sum()
    if nulos.sum() > 0:
        print(nulos[nulos > 0])
    else:
        print("Sem valores ausentes")


Sales Pipeline:
engage_date     500
close_date     2089
close_value    2089
dtype: int64

Products:
Sem valores ausentes

Sales Teams:
Sem valores ausentes

Accounts:
subsidiary_of    70
dtype: int64


### OBSERVAÇÕES:

**Problemas Identificados:**
- close_value com valores ausentes em deals não fechados
- close_date ausente em deals em andamento (Prospecting/Engaging)
- Contas não identificadas


## LIMPEZA E TRATAMENTO DE DADOS

###  Conversão de Tipos de Dados

In [27]:
# Converter colunas de data
sales_pipeline['engage_date'] = pd.to_datetime(sales_pipeline['engage_date'])
sales_pipeline['close_date'] = pd.to_datetime(sales_pipeline['close_date'])

# Verificar conversão
print(f"\nTipo de engage_date: {sales_pipeline['engage_date'].dtype}")
print(f"Tipo de close_date: {sales_pipeline['close_date'].dtype}")


Tipo de engage_date: datetime64[ns]
Tipo de close_date: datetime64[ns]


### Tratamento de Valores Ausentes

In [28]:
# Análise de close_value ausente
print("Análise de close_value ausente:\n")
print(sales_pipeline['deal_stage'].value_counts())

# Preencher close_value = 0 para deals Lost
sales_pipeline.loc[sales_pipeline['deal_stage'] == 'Lost', 'close_value'] = \
    sales_pipeline.loc[sales_pipeline['deal_stage'] == 'Lost', 'close_value'].fillna(0)

print("\n close_value preenchido para deals Lost")

Análise de close_value ausente:

deal_stage
Won            4238
Lost           2473
Engaging       1589
Prospecting     500
Name: count, dtype: int64

 close_value preenchido para deals Lost


### Merge das Tabelas

Criação do **df_master** unindo todas as informações.

In [29]:
# Merge 1: Sales Pipeline + Products
df_master = pd.merge(
    sales_pipeline, 
    products, 
    on='product', 
    how='left'
)
print(f"Merge 1: {df_master.shape}")

# Merge 2: + Sales Teams
df_master = pd.merge(
    df_master,
    sales_teams,
    on='sales_agent',
    how='left'
)
print(f"Merge 2: {df_master.shape}")

# Merge 3: + Accounts
df_master = pd.merge(
    df_master,
    accounts,
    on='account',
    how='left'
)
print(f"Merge 3 (final): {df_master.shape}")

# Verificar resultado
print(f"\nDataFrame Master: {df_master.shape[0]:,} linhas × {df_master.shape[1]} colunas")
display(df_master.head())

Merge 1: (8800, 10)
Merge 2: (8800, 12)
Merge 3 (final): (8800, 18)

DataFrame Master: 8,800 linhas × 18 colunas


,opportunity_id,sales_agent,product,account,deal_stage,engage_date,close_date,close_value,series,sales_price,manager,regional_office,sector,year_established,revenue,employees,office_location,subsidiary_of
0,1C1I7A6R,Moses Frase,GTX Plus Basic,Cancity,Won,2016-10-20,2017-03-01,1054.0,GTX,1096.0,Dustin Brinkmann,Central,retail,2001.0,718.62,2448.0,United States,NaN
1,Z063OYW0,Darcel Schlecht,GTXPro,Isdom,Won,2016-10-25,2017-03-11,4514.0,NaN,NaN,Melvin Marxen,Central,medical,2002.0,3178.24,4540.0,United States,NaN
2,EC4QE1BX,Darcel Schlecht,MG Special,Cancity,Won,2016-10-25,2017-03-07,50.0,MG,55.0,Melvin Marxen,Central,retail,2001.0,718.62,2448.0,United States,NaN
3,MV1LWRNH,Moses Frase,GTX Basic,Codehow,Won,2016-10-25,2017-03-09,588.0,GTX,550.0,Dustin Brinkmann,Central,software,1998.0,2714.90,2641.0,United States,Acme Corporation
4,PE84CX4O,Zane Levy,GTX Basic,Hatfan,Won,2016-10-25,2017-03-02,517.0,GTX,550.0,Summer Sewald,West,services,1982.0,792.46,1299.0,United States,NaN



## FEATURE ENGINEERING

Criação de novas variáveis para análise.

### Ciclo de Vendas (em dias)

In [30]:
# Calcular ciclo de vendas
df_master['ciclo_dias'] = (df_master['close_date'] - df_master['engage_date']).dt.days

print(f"\nCiclo médio: {df_master['ciclo_dias'].mean():.1f} dias")
print(f"Ciclo mínimo: {df_master['ciclo_dias'].min()} dias")
print(f"Ciclo máximo: {df_master['ciclo_dias'].max()} dias")


Ciclo médio: 48.0 dias
Ciclo mínimo: 1.0 dias
Ciclo máximo: 138.0 dias


### Taxa de Conversão

In [31]:
# Criar flag binária de conversão
df_master['converteu'] = (df_master['deal_stage'] == 'Won').astype(int)

print("Converteu (0/1)")
print(f"\nTaxa de conversão: {df_master['converteu'].mean()*100:.2f}%")

Converteu (0/1)

Taxa de conversão: 48.16%


### Categoria de Ticket

Classificação do valor da venda em categorias.

In [32]:
# Criar categorias de ticket
def categorizar_ticket(valor):
    if pd.isna(valor) or valor == 0:
        return 'Sem Valor'
    elif valor < 1000:
        return 'Baixo'
    elif valor < 3000:
        return 'Médio'
    elif valor < 10000:
        return 'Alto'
    else:
        return 'Premium'

df_master['categoria_ticket'] = df_master['close_value'].apply(categorizar_ticket)
print(df_master['categoria_ticket'].value_counts())

categoria_ticket
Sem Valor    4562
Baixo        1857
Alto         1780
Médio         586
Premium        15
Name: count, dtype: int64


### Extração de Informações Temporais

In [33]:
# Extrair ano, mês, trimestre
df_master['ano'] = df_master['engage_date'].dt.year
df_master['mes'] = df_master['engage_date'].dt.month
df_master['trimestre'] = df_master['engage_date'].dt.quarter
df_master['mes_ano'] = df_master['engage_date'].dt.to_period('M').astype(str)

print(df_master['ano'].value_counts().sort_index())

ano
2016.0     358
2017.0    7942
Name: count, dtype: int64


### RESUMO DAS FEATURES CRIADAS:


 `ciclo_dias`  Numérico  Dias entre engage e close 

 `converteu`  Binário  1 = Won, 0 = Outros 

 `categoria_ticket`  Categórico  Baixo/Médio/Alto/Premium 

 `ano`  Numérico  Ano do engajamento 

 `mes`  Numérico  Mês (1-12) 

 `trimestre`  Numérico  Trimestre (1-4)

 `mes_ano`  String  Formato YYYY-MM 

8 novas features criadas!



### Métricas Gerais do Negócio

In [34]:
# KPIs principais
total_deals = len(df_master)
deals_won = df_master['converteu'].sum()
taxa_conversao = (deals_won / total_deals) * 100
receita_total = df_master[df_master['deal_stage'] == 'Won']['close_value'].sum()
ticket_medio = df_master[df_master['deal_stage'] == 'Won']['close_value'].mean()
ciclo_medio = df_master[df_master['deal_stage'] == 'Won']['ciclo_dias'].mean()


print(f"\nTotal de Deals:        {total_deals:,}")
print(f"Deals Ganhos (Won):    {deals_won:,}")
print(f"Taxa de Conversão:     {taxa_conversao:.2f}%")
print(f"Receita Total:         ${receita_total:,.2f}")
print(f"Ticket Médio:          ${ticket_medio:,.2f}")
print(f"Ciclo Médio de Venda:  {ciclo_medio:.1f} dias")



Total de Deals:        8,800
Deals Ganhos (Won):    4,238
Taxa de Conversão:     48.16%
Receita Total:         $10,005,534.00
Ticket Médio:          $2,360.91
Ciclo Médio de Venda:  51.8 dias



##  PREPARAÇÃO PARA SQL

### Exportar Dataset Limpo

In [35]:
# Exportar df_master limpo
df_master.to_csv('sales_data_clean.csv', index=False)

print(f"{df_master.shape[0]:,} registros × {df_master.shape[1]} colunas")
print("\nColunas incluídas:")
print(list(df_master.columns))

8,800 registros × 25 colunas

Colunas incluídas:
['opportunity_id', 'sales_agent', 'product', 'account', 'deal_stage', 'engage_date', 'close_date', 'close_value', 'series', 'sales_price', 'manager', 'regional_office', 'sector', 'year_established', 'revenue', 'employees', 'office_location', 'subsidiary_of', 'ciclo_dias', 'converteu', 'categoria_ticket', 'ano', 'mes', 'trimestre', 'mes_ano']


### Tabelas Dimensionais

Criando CSVs para importação no PostgreSQL (modelo Star Schema).

In [36]:
# 1. dim_products
dim_products = df_master[['product', 'series', 'sales_price']].drop_duplicates()
dim_products.to_csv('dim_products.csv', index=False)
print(f"dim_products.csv: {len(dim_products)} registros")

# 2. dim_sales_teams  
dim_sales_teams = df_master[['sales_agent', 'manager', 'regional_office']].drop_duplicates()
dim_sales_teams = dim_sales_teams[dim_sales_teams['sales_agent'].notna()]
dim_sales_teams.to_csv('dim_sales_teams.csv', index=False)
print(f"dim_sales_teams.csv: {len(dim_sales_teams)} registros")

# 3. dim_accounts
dim_accounts = df_master[['account', 'sector', 'year_established', 'revenue', 
                           'employees', 'office_location', 'subsidiary_of']].drop_duplicates()
dim_accounts = dim_accounts[dim_accounts['account'] != 'Não Identificado']
dim_accounts['year_established'] = dim_accounts['year_established'].astype('Int64')
dim_accounts['employees'] = dim_accounts['employees'].astype('Int64')
dim_accounts.to_csv('dim_accounts.csv', index=False)
print(f"dim_accounts.csv: {len(dim_accounts)} registros")

# 4. fact_sales_pipeline
fact_sales = df_master[['opportunity_id', 'sales_agent', 'product', 'account',
                         'deal_stage', 'engage_date', 'close_date', 'close_value']].copy()
fact_sales['account'] = fact_sales['account'].replace('Não Identificado', None)
fact_sales.to_csv('fact_sales_pipeline.csv', index=False)
print(f"fact_sales_pipeline.csv: {len(fact_sales)} registros")


dim_products.csv: 7 registros
dim_sales_teams.csv: 30 registros
dim_accounts.csv: 85 registros
fact_sales_pipeline.csv: 8800 registros



##  CONCLUSÕES E INSIGHTS

#### ** Performance Geral -
-  Taxa de conversão de **48.16%** 
-  Receita total de **$10.01M** em 15 meses
-  Ticket médio de **$2,361** por venda
-  Ciclo médio de **52 dias** 


#### ** SQL/PostgreSQL:**
- Modelagem dimensional (Star Schema)
- 4 tabelas criadas com relacionamentos
- 4 views analíticas para Power BI
- 7 análises SQL complexas

#### ** Power BI:**
- Dashboard 1: Overview Executivo
- Dashboard 2: Análise de Produtos
- Dashboard 3: Funil de Vendas
- Dashboard 4: Performance de Gerentes


###  Documentação Técnica

**Ferramentas Utilizadas:**
- Python 3.x (Pandas, NumPy, Matplotlib, Seaborn)
- PostgreSQL 17
- Power BI Desktop

**Arquivos Gerados:**
- `sales_data_clean.csv` (dataset master limpo)
- 4 CSVs dimensionais para SQL
- Scripts SQL 
- 4 Dashboards interativos (.pbix)


### Autor

**[Tainá Silva]**  
Data Analyst | Python | SQL | Power BI  
[LinkedIn] | [GitHub] | [email]

*Projeto desenvolvido como parte do portfólio de análise de dados*
